# Random Forest / XGBoost comparison

Both classifiers use the same saved embeddings, feature columns, and seed 42.
Random Forest uses `weight ** 0.75` and XGBoost uses `weight ** 0.35`.
The default embedding is GloVe. Both classifiers are fitted again on the same training rows;
RF predictions remain in memory, and only XGBoost predictions are written to its own directory.
`Scorer` reports accuracy, support-weighted precision/F1, MAE, and RMSE on all three splits.
The 184-round XGBoost defaults were selected by accuracy on a holdout from the GloVe training pool.
The reference test accuracy of 40.26% was measured with native CUDA XGBoost and 600-dimensional GloVe.
This notebook trains with SparkXGBClassifier and scores the embeddings loaded for this run.

Install the dependencies in `requirements.txt` and restart the kernel after changing packages or model code.
The official XGBoost Spark integration is tested on Linux; Windows compatibility must be checked
in your environment: https://xgboost.readthedocs.io/en/stable/tutorials/spark_estimator.html

`REBUILD=False` reuses `data/processed/embeddings/glove/`. Set it to `True` only when inputs
must be rebuilt. Override classifier constructor values through `para` (XGBoost) and `rf_para`.
The classifier never uses external validation/test rows for fitting or early stopping.
Choose hyperparameters on training folds; reserve test scores for final reporting.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild inputs and the selected embeddings, then save those results.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False

# Override model hyperparameters here; an empty dictionary uses the defaults.
para = {}
rf_para = {}

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = (
    SparkSession.builder 
    .master("local[4]")
    .appName("DrugReviewXGBoostComparison")
    .config("spark.driver.memory", "4g")
    .config("spark.default.parallelism", "4")
    .config("spark.task.cpus", "2")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split(df_engineered, SEED)
    df_train = weight(df_train)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)


In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import fit_phrases

USE_EMBEDDINGS = {"glove": True}
embedding_names = [name for name, enabled in USE_EMBEDDINGS.items() if enabled]

train_embedding_dict = {}
valid_embedding_dict = {}
test_embedding_dict = {}

if REBUILD:
    phrases = fit_phrases(df_train) if "glove" in embedding_names else None
    embedding_model = EmbeddingModel(df_train, df_valid, df_test, phrases)
    for name in embedding_names:
        print(f"Embedding Model {name} Training")
        if name == "tfidf":
            embedding_model.baseline_train(model_name=name)
            embedding_frames = embedding_model.baseline_predict(model_name=name)
        else:
            EmbeddingModel.train_registry[name](embedding_model)
            embedding_frames = EmbeddingModel.predict_registry[name](embedding_model)
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = embedding_frames
        data_oi.save_embedding_data(
            name,
            train_embedding_dict[name],
            valid_embedding_dict[name],
            test_embedding_dict[name],
        )
else:
    for name in embedding_names:
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = (
            data_oi.load_embedding_data(name)
        )

print(f"Embeddings in this run: {embedding_names}")


In [ ]:
from src.analytics import AnalyticsModel

analytics_names = ("random_forest", "xgboost")
model_params = {"random_forest": rf_para, "xgboost": para}


In [ ]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}

for embedding_name in embedding_names:
    for analytics_name in analytics_names:
        model_name = f"{analytics_name}__{embedding_name}"
        print(f"Running {model_name}")
        instance = AnalyticsModel(
            embedding_name,
            train_embedding_dict[embedding_name],
            valid_embedding_dict[embedding_name],
            test_embedding_dict[embedding_name],
        )
        predictions = AnalyticsModel.registry[analytics_name](
            instance, para=model_params[analytics_name]
        )
        # Materialize once before scoring and releasing assembled features.
        for frame in predictions:
            frame.cache().count()
        train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = predictions
        instance.release_features()
        if analytics_name == "xgboost":
            data_oi.save_prediction_data(model_name, *predictions)


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)

In [ ]:
# Run after inspecting the score tables to release prediction caches.
data_oi.unpersist(
    *train_pred_dict.values(), *valid_pred_dict.values(), *test_pred_dict.values()
)
